<a href="https://colab.research.google.com/github/jotokx88/Data-visualization/blob/main/Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install dash yfinance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 38.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [dash]


In [16]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import yfinance as yf
import plotly.express as px
import pandas as pd

In [20]:
app = dash.Dash(__name__)
app.layout = html.Div([
    dcc.Dropdown(
        id='stock-selector',
        options=[
            {'label': 'AAPL', 'value': 'AAPL'},
            {'label': 'TSLA', 'value': 'TSLA'},
            {'label': 'MSFT', 'value': 'MSFT'}
        ],
        multi=True,
        value='AAPL'
    ),
    dcc.DatePickerRange(
        id='date-picker',
        min_date_allowed='2020-01-01',
        max_date_allowed='2025-12-31',
        start_date='2022-06-01',
        end_date='2024-06-01'
    ),
    dcc.Graph(id='stock-price-chart')
])

In [21]:
@app.callback(
    Output('stock-price-chart', 'figure'),
    [Input('stock-selector', 'value')]
)
def update_stock_price_chart(selected_stock):
    stock_data = yf.download(selected_stock, start='2022-06-01', end='2024-06-01')
    fig = px.line(stock_data, x=stock_data.index, y='Close', title=f'{selected_stock} Stock Price analysis')
    return fig

In [22]:
@app.callback(
    Output('stock-price-chart', 'figure'),
    [Input('stock-selector', 'value'),
     Input('date-picker', 'start_date'),
     Input('date-picker', 'end_date')]
)
def update_stock_price_chart(selected_stock, start_date, end_date):
    try:
        # Read local stocks data
        df = pd.read_csv('/content/stocks.csv')
        # Filter data for the selected stock
        filtered_df = df[df['Ticker'] == selected_stock]
        # Ensure Date column is in datetime format for filtering
        filtered_df['Date'] = pd.to_datetime(filtered_df['Date'])
        # Filter by selected date range
        if start_date:
            filtered_df = filtered_df[filtered_df['Date'] >= pd.to_datetime(start_date)]
        if end_date:
            filtered_df = filtered_df[filtered_df['Date'] <= pd.to_datetime(end_date)]
        # Generate interactive Plotly line chart
        fig = px.line(
            filtered_df,
            x='Date',
            y='Close',
            title=f'{selected_stock} Stock Price Analysis'
        )
        return fig
    except Exception as e:
        # Return empty figure on error
        return px.line(title=f'Error loading data: {str(e)}')

In [26]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd

# Initialize the Dash application
app = dash.Dash(__name__)

# Define the Layout with dropdown and DatePickerRange
app.layout = html.Div([
    html.H1("Stock Price Viewer", style={'textAlign': 'center'}),

    html.Label("Select Stock Ticker:"),
    dcc.Dropdown(
        id='stock-selector',
        options=[
            {'label': 'AAPL', 'value': 'AAPL'},
            {'label': 'TSLA', 'value': 'TSLA'},
            {'label': 'MSFT', 'value': 'MSFT'}
        ],
        multi=True,
        value=['AAPL'],
        style={'width': '50%', 'marginBottom': '20px'}
    ),

    html.Label("Select Date Range:"),
    html.Div([
        dcc.DatePickerRange(
            id='date-picker',
            min_date_allowed='2020-01-01',
            max_date_allowed='2025-12-31',
            start_date='2022-06-01',
            end_date='2024-06-01'
        )
    ], style={'marginBottom': '20px'}),

    dcc.Graph(id='stock-price-chart')
])

# Define Callback to update the graph based on selection
@app.callback(
    Output('stock-price-chart', 'figure'),
    [Input('stock-selector', 'value'),
     Input('date-picker', 'start_date'),
     Input('date-picker', 'end_date')]
)
def update_stock_price_chart(selected_stocks, start_date, end_date):
    try:
        # Read local stocks data
        df = pd.read_csv('/content/stocks.csv')

        # Ensure we have a list of selected stocks
        if not selected_stocks:
            return px.line(title='Please select at least one stock ticker.')

        if isinstance(selected_stocks, str):
            selected_stocks = [selected_stocks]

        # Filter data for the selected stocks using .isin()
        filtered_df = df[df['Ticker'].isin(selected_stocks)].copy()

        # Ensure Date column is in datetime format for filtering
        filtered_df['Date'] = pd.to_datetime(filtered_df['Date'])

        # Filter by selected date range
        if start_date:
            filtered_df = filtered_df[filtered_df['Date'] >= pd.to_datetime(start_date)]
        if end_date:
            filtered_df = filtered_df[filtered_df['Date'] <= pd.to_datetime(end_date)]

        # Sort by date to maintain chronological order
        filtered_df = filtered_df.sort_values('Date')

        # Generate interactive Plotly line chart grouped by Ticker
        fig = px.line(
            filtered_df,
            x='Date',
            y='Close',
            color='Ticker',
            title='Stock Price Analysis'
        )
        return fig
    except Exception as e:
        # Return empty figure with error message on failure
        return px.line(title=f'Error loading data: {str(e)}')

# Run the Dash server inline in Colab
if __name__ == '__main__':
    app.run(jupyter_mode='inline', port=8050)

<IPython.core.display.Javascript object>

In [28]:
import ipywidgets
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

In [30]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

# Generate random uniform data
x = np.random.uniform(0.1, 10.0, size=100)
# Generate matching random normal noise
noise = np.random.normal(0, 1, size=100)

def plot_fct(w=1):
    y = np.log(x) + w * noise
    plt.figure(figsize=(8, 5))
    plt.scatter(x, y, alpha=0.7, edgecolors='k')
    plt.title(f'Log plot with noise weight w={w}')
    plt.xlabel('x')
    plt.ylabel('y = log(x) + w * noise')
    plt.grid(True)
    plt.show()

# Create interactive widget
widgets.interact(plot_fct, w=widgets.FloatSlider(min=0.0, max=5.0, step=0.1, value=1.0))

interactive(children=(FloatSlider(value=1.0, description='w', max=5.0), Output()), _dom_classes=('widget-inter…

<function __main__.plot_fct(w=1)>

In [31]:
from sklearn.datasets import make_moons

In [33]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from sklearn.datasets import make_moons

def plot_moons(samples=200, noise=0.1):
    X, y = make_moons(n_samples=samples, noise=noise, random_state=0)
    plt.figure(figsize=(8, 5))
    plt.scatter(X[y == 0, 0], X[y == 0, 1], color='red', label='Class 0', alpha=0.7)
    plt.scatter(X[y == 1, 0], X[y == 1, 1], color='blue', label='Class 1', alpha=0.7)
    plt.title(f"Make Moons Dataset (Samples: {samples}, Noise: {noise})")
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.legend()
    plt.grid(True)
    plt.show()

# Create interactive dashboard control
widgets.interact(
    plot_moons,
    samples=widgets.IntSlider(min=50, max=1000, step=50, value=200),
    noise=widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.1)
)

interactive(children=(IntSlider(value=200, description='samples', max=1000, min=50, step=50), FloatSlider(valu…

<function __main__.plot_moons(samples=200, noise=0.1)>